In [ ]:
import uproot
import awkward as ak
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import math

file_name = '../../MCEventInfoForBinningOptimisation.root'

with uproot.open(file_name) as f:
    # Load some data from the file
    df_events = f['MCEvent'].arrays([
                                    'TrueENu',
                                    'TrueCZ',
                                    'TrueEHad',
                                    'TrueELep',
                                    'RecoENu',
                                    'RecoEHad',
                                    'RecoELep',
                                    'RecoCZ',
                                    'Sample',
                                    'EventWeight'
                                    ],
                                     library='pd')

df_events['RecoY'] = df_events['RecoEHad'] / df_events['RecoENu']

samples = {
            'FC_numu': 0,
            'FC_nue': 1,
            'PC_numu': 2,
            'PC_nue': 3,
            }

display(df_events)


In [ ]:
def create_energy_bins(cp_step, m32_step, bsm_step):

    delta_cp_bins = np.arange(0., 1., cp_step)
    mass_ordering_bins = np.arange(1, 10, m32_step)
    bsm_bins = np.arange(10, 100+bsm_step, bsm_step)

    E_nu_bins = np.concatenate([delta_cp_bins,mass_ordering_bins,bsm_bins])
    return E_nu_bins

E_nu_bins = create_energy_bins(0.2, 0.05, 0.5)
#print(f"{len(E_nu_bins)}: {E_nu_bins}")

In [ ]:
# setup bins and dataframes for each sample
def setup_dfs(samples):
    dfs=dict.fromkeys(samples.keys())
    for sample, sample_id in samples.items():
        dfs[sample]=df_events[df_events['Sample']==sample_id].copy()
        dfs[sample]=dfs[sample].reset_index(drop=True)
    return dfs

def setup_bins(samples):
    bins=dict.fromkeys(samples.keys())
    for sample, sample_id in samples.items():
        # add overflow bin to reproduce numpy histogram behaviour
        # TODO check if inf is already present
        axis_bins=dict.fromkeys(histo_axes_bins_initial.keys())
        for axis, edges in histo_axes_bins_initial.items():
            axis_bins[axis] = np.concatenate([edges, [float('inf')]])
            #print(f"Initial {axis} edges for {sample}: {axis_digi_bins[axis]}")
            samples_dfs[sample][axis+'binIndex'] = np.nan
        bins[sample]=axis_bins
    return bins

###
def get_bin_indeces(df, axis, bins):
    mc_counts=[]
    data_counts=[]

    data=df[axis].to_numpy()
    weights=df['EventWeight'].to_numpy()

    indeces = np.digitize(data, bins, right=False)

    for bin in range(1,len(bins)):
        mc_c = np.count_nonzero(indeces == bin)
        mc_counts.append(mc_c)

        bin_weigths=weights[indeces == bin]
        data_c=np.sum(bin_weigths)*exposure_scaling
        data_counts.append(data_c)
    return indeces, mc_counts, data_counts
###

###
def check_bins_are_ok_axis(axis, bins, min_mc, min_data, df, idx_sel):

    new_bins = bins.copy()
    
    # only used for indeces computation
    df_sel=df.iloc[idx_sel]

    indeces, mc_counts, data_counts = get_bin_indeces(df_sel, axis, bins)

    df.iloc[idx_sel, df.columns.get_loc(axis+'binIndex')]=indeces
    
    n_bins=len(mc_counts[:-1])

    ok=False
    bad_edges=[]
    bin_idx = 0
    while bin_idx < n_bins:
        mc_count=mc_counts[bin_idx]
        data_count=data_counts[bin_idx]
        while True:
            if mc_count < min_mc or data_count < min_data:
                #print(f"bad bin: {bin_idx} {mc_count}, {data_count}")

                # last bin, special
                if bin_idx == n_bins-1:
                    # actually need to merge with last good bin
                    if len(new_bins) < 4:
                        raise RuntimeError(f"all bins are bad! data sum {np.sum(data_counts[:-1])} mc sum {np.sum(mc_counts[:-1])}")
                    else:
                        bad_edge=new_bins[::-1][2]
                        new_bins=np.delete(new_bins, np.where(new_bins == bad_edge))
                        bad_edges.append(bad_edge)
                        bin_idx += 1
                        break
                else:
                    bin_idx += 1
                    bad_edge=bins[bin_idx]
                    new_bins=np.delete(new_bins, np.where(new_bins == bad_edge))
                    bad_edges.append(bad_edge)
                    mc_count=mc_count+mc_counts[bin_idx]
                    data_count=data_count+data_counts[bin_idx]
                    #print(f"merged {mc_count}, {data_count}")
            else:
                bin_idx += 1
                break
    
    #if not ok:    
        #print(f"bad bins: {bad_bins}")
       
        #print(f"mc counts: {mc_counts[:-1]}")

    data_sum=np.sum(data_counts[:-1])
    mc_sum=np.sum(mc_counts[:-1])
    print(f"edges: {bins[:-1]}")
    print(f"data counts: {data_counts[:-1]}, data sum: {data_sum}")
    print(f"mc counts: {mc_counts[:-1]}, mc sum:  {mc_sum}")

    if (len(new_bins)==len(bins)):
        ok=True
    else:
        print(f"removed edges: {bad_edges}")
        print(f"new {axis} edges: {new_bins}")
        if (len(new_bins)==3):
            #print(f"[WARNING] axis {axis} fully merged! data sum {data_sum} mc sum {mc_sum}")
            raise RuntimeError(f"[WARNING] axis {axis} fully merged!")
            
    return ok, new_bins#, bad_bins_mc_counts, bad_bins_data_counts
###

###
def do_optimise_bins_work(outer_axes, axis, min_mc, min_data, df, sample, idx_sel):
    
    while True:
        axis_bins=samples_bins[sample][axis]
        ok, new_bins = check_bins_are_ok_axis(axis, 
                                                      axis_bins,
                                                      min_mc,
                                                      min_data,
                                                      df,
                                                      idx_sel)
        if ok:
            #print(f"{outer_axes} {axis} ok")
            break
        else:
            print(f"{outer_axes} {axis} were optimised")
            samples_bins[sample][axis]=new_bins
    return True
###

###
def optimise_bins_loop(outer_bins, outer_axes, axis, min_mc, min_data, df, sample, indices=()):
    if not outer_bins:

        true_array = [True] * len(df)

        bin_mask=pd.Series(true_array, index=df.index)
        bin_indices =  [idx+1 for idx in indices]
        for i, bin_idx in enumerate(bin_indices):
            o_a=outer_axes[i]
            bin_mask=bin_mask & (df[o_a+'binIndex'] == bin_idx)
            #print(f"masking {o_a} bin {bin_idx}")

        sel_idx = df.index[ bin_mask ]
        outer_axes_str=f"{outer_axes} bin {bin_indices}"
        print(f"checking {outer_axes_str} if {axis} bins are ok  with total: {len(sel_idx)}")
        
        ok = do_optimise_bins_work(outer_axes_str, axis, min_mc, min_data, df, sample, sel_idx)

        return
    for i in range(outer_bins[0]):
        optimise_bins_loop(outer_bins[1:], outer_axes, axis, min_mc, min_data, df, sample, indices + (i,))
###

def optimise_bins(sample, min_mc, min_data):
    df = samples_dfs[sample]

    for i, axis in enumerate(axes):
        axes_to_opt=axes[:i+1]

        bins=samples_bins[sample]
        bins_array=list(bins.values())
        
        bins_to_opt=bins_array[:i+1]
        #print(f"optimising {axes_to_opt}")
        target_bins=bins_array[i]

        #print(f"target: {axis} {target_bins}")

        outer_axes=axes_to_opt
        outer_axes.pop()

        outer_bins=bins_to_opt
        outer_bins.pop()

        outer_n_bins = [len(x)-2 for x in outer_bins]

        dim_factor=1
        if (len(outer_axes) == 1):
            dim_factor=10

        optimise_bins_loop(outer_n_bins, outer_axes, axis, min_mc, min_data*dim_factor, df, sample)


# algo parameters
min_mc_events=100
min_data_events=3

# 1. is 400 kt-y
exposure_scaling=1

# order of axes here determines optimisation order
histo_axes_bins_initial = {
                        'RecoCZ': np.arange(-1., 1.01, 0.2),
                        'RecoENu': E_nu_bins,
                        'RecoY': np.arange(0., 1.01, 0.1),
                        }

axes=list(histo_axes_bins_initial.keys())

samples_dfs=setup_dfs(samples)
samples_bins=setup_bins(samples)

for sample, sample_id in samples.items():
    #optimise_sample_bins(sample)
    optimise_bins(sample, min_mc_events, min_data_events)
    print("")

print(f"exposure {400*exposure_scaling} kt-y")

for sample in samples.keys():
    for axis, edges in samples_bins[sample].items():
        print(f"Final {sample} {axis} edges: {edges[:-1]}")